# Model Training

Training and comparing three classifiers - Logistic Regression as a baseline, Random Forest and XGBoost - on the engineered features. 

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_ingestion import DataIngestion
from src.data_validation import DataValidation
from src.data_transformation import DataTransformation
from src.model_trainer import ModelTrainer
from src.utils import read_csv_safely

In [2]:
ingestion = DataIngestion()
ingestion.config.raw_data_path = '../data/train.csv'
ingestion.config.train_data_path = '../artifacts/train.csv'
ingestion.config.test_data_path = '../artifacts/test.csv'

train_path, test_path = ingestion.initiate_data_ingestion()
print(train_path, test_path)

2026-07-18 11:52:26,133 | src.data_ingestion | INFO | Starting data ingestion
2026-07-18 11:52:26,460 | src.data_ingestion | INFO | Raw dataset shape: (58592, 44)
2026-07-18 11:52:27,280 | src.data_ingestion | INFO | Train shape: (46873, 44), Test shape: (11719, 44)


../artifacts/train.csv ../artifacts/test.csv


In [3]:
train_df = read_csv_safely(train_path)
test_df = read_csv_safely(test_path)
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

Train shape: (46873, 44)
Test shape: (11719, 44)


In [4]:
validator = DataValidation()
print('Train data valid:', validator.run_all_checks(train_df))
print('Test data valid:', validator.run_all_checks(test_df))

2026-07-18 11:52:27,953 | src.data_validation | INFO | Validation passed: True
2026-07-18 11:52:28,015 | src.data_validation | INFO | Validation passed: True


Train data valid: True
Test data valid: True


In [5]:
transformer = DataTransformation(artifacts_dir='../artifacts')
X_train, y_train = transformer.fit_transform(train_df)
X_test = transformer.transform(test_df.drop(columns=['is_claim']))
y_test = test_df['is_claim']
print(X_train.shape, X_test.shape)

2026-07-18 11:52:28,035 | src.data_transformation | INFO | Fitting data transformation pipeline
2026-07-18 11:52:28,576 | src.data_transformation | INFO | Final feature matrix shape: (46873, 96)


(46873, 96) (11719, 96)


### Class imbalance check


In [6]:
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / pos
print(f'Positive samples: {pos}, Negative samples: {neg}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

Positive samples: 2998, Negative samples: 43875
scale_pos_weight: 14.63


### Training all three candidate models


In [7]:
trainer = ModelTrainer(artifacts_dir='../artifacts')
best_name, best_model, results = trainer.train_and_evaluate(
    X_train, y_train, X_test, y_test
)

2026-07-18 11:52:28,808 | src.model_trainer | INFO | Training logistic_regression
2026-07-18 11:52:30,169 | src.model_trainer | INFO | logistic_regression -> ROC-AUC: 0.5862, F1: 0.1398
2026-07-18 11:52:30,169 | src.model_trainer | INFO | Training random_forest
2026-07-18 11:52:33,740 | src.model_trainer | INFO | random_forest -> ROC-AUC: 0.6484, F1: 0.1661
2026-07-18 11:52:33,740 | src.model_trainer | INFO | Training xgboost
2026-07-18 11:52:35,237 | src.model_trainer | INFO | xgboost -> ROC-AUC: 0.6436, F1: 0.1613
2026-07-18 11:52:35,241 | src.model_trainer | INFO | Best model: random_forest


In [8]:
results_df = pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)
results_df

,roc_auc,f1_score
random_forest,0.6484,0.1661
xgboost,0.6436,0.1613
logistic_regression,0.5862,0.1398


print('Best model selected:', best_name)
print('Saved to artifacts/model.pkl')
best_model

## Summary

- Logistic Regression gives a reasonable baseline but is clearly outperformed by the tree based models, confirming the non-linear relationships seen in feature selection.
- Random Forest and XGBoost land close to each other on ROC-AUC, and the better of the two is saved automatically.
- Given how imbalanced the target is, absolute scores here look modest - this is expected and typical for real world insurance claim data, where claims genuinely are hard to predict from policy/vehicle attributes alone.

